# Input Data Overview

A quick inventory of everything under `data/input/`: for each file, the
first five rows plus basic metadata (rows, columns, time range, missing
values). See {doc}`../markdown/data_sources` for where the data comes from
and what each file is meant for.

The MaStR solar table has ~6.3M rows, so metadata is read column-by-column
via `pyarrow` rather than loading every file fully into memory.

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import xarray as xr

from hpsp.paths import ProjPaths

paths = ProjPaths()
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 200)


def head(path, n=5):
    """First `n` rows of a (flat, index-free) parquet file, without reading it all."""
    return next(pq.ParquetFile(path).iter_batches(batch_size=n)).to_pandas()


def file_info(path):
    pf = pq.ParquetFile(path)
    return {
        "file": str(path.relative_to(paths.input_path)),
        "size_MB": round(path.stat().st_size / 1e6, 1),
        "rows": pf.metadata.num_rows,
        "columns": len(pf.schema_arrow.names),
    }


def timeseries_meta(df):
    """Metadata for a timestamp-indexed capacity-factor frame."""
    idx = df.index
    steps = pd.Series(idx).diff().value_counts()
    return pd.Series(
        {
            "start": idx.min(),
            "end": idx.max(),
            "rows": len(idx),
            "years": idx.year.nunique(),
            "time step (most common)": steps.index[0],
            "irregular steps": int(steps.iloc[1:].sum()),
            "duplicate timestamps": int(idx.duplicated().sum()),
            "columns": df.shape[1],
            "missing values (total)": int(df.isna().sum().sum()),
            "columns with any NaN": int(df.isna().any().sum()),
            "value min": float(df.min().min()),
            "value max": float(df.max().max()),
        },
        name="value",
    ).to_frame()

## Summary of all files

In [2]:
parquet_files = sorted(paths.input_path.rglob("*.parquet"))
summary = pd.DataFrame([file_info(p) for p in parquet_files])
nc_files = sorted(paths.input_path.rglob("*.nc"))
for p in nc_files:
    with xr.open_dataset(p) as ds:
        summary.loc[len(summary)] = {
            "file": str(p.relative_to(paths.input_path)),
            "size_MB": round(p.stat().st_size / 1e6, 1),
            "rows": None,
            "columns": None,
        }
summary

,file,size_MB,rows,columns
0,mastr/solar.parquet,211.6,6304840,22
1,mastr/solar_technical_detail.parquet,74.8,6304840,3
2,mastr/wind.parquet,1.5,43262,20
3,pecd/pecd_country_capacity_factors_simple.parquet,231.8,403247,110
4,pecd/pecd_country_capacity_factors_simple_de.p...,11.9,403247,4
5,pecd/pecd_solar_capacity_factors.parquet,51.6,96432,153
6,pecd/pecd_wind_offshore_capacity_factors.parquet,2.9,96432,7
7,pecd/pecd_wind_onshore_capacity_factors.parquet,2.3,96432,8
8,regions/lau_nuts_correspondence.parquet,0.1,10980,2
9,pecd/peof_region_mask.nc,34.7,None,None


## PECD capacity factors: national, Germany (Track B)

`pecd/pecd_country_capacity_factors_simple_de.parquet` -- hourly, one
column per technology. Solar is already a blend of the 4 PV
sub-technologies (weighted by Germany's market mix).

In [3]:
cf_de = pd.read_parquet(paths.pecd_capacity_factors_national_de)
cf_de.head()

technology,wind_onshore,wind_offshore,solar
timestamp,,,
1980-01-01 00:00:00,0.467822,0.318317,0.0
1980-01-01 01:00:00,0.456618,0.273357,0.0
1980-01-01 02:00:00,0.458957,0.269950,0.0
1980-01-01 03:00:00,0.444485,0.317110,0.0
1980-01-01 04:00:00,0.436744,0.406120,0.0


In [4]:
timeseries_meta(cf_de)

,value
start,1980-01-01 00:00:00
end,2025-12-31 22:00:00
rows,403247
years,46
time step (most common),0 days 01:00:00
irregular steps,0
duplicate timestamps,0
columns,3
missing values (total),0
columns with any NaN,0


In [5]:
cf_de.describe().T

,count,mean,std,min,25%,50%,75%,max
technology,,,,,,,,
wind_onshore,403247.0,0.236498,0.197625,0.000111,0.083628,0.174648,0.337616,0.846755
wind_offshore,403247.0,0.430347,0.280804,0.000000,0.177927,0.396857,0.673528,0.952393
solar,403247.0,0.107017,0.159064,0.000000,0.000000,0.003306,0.180608,0.718598


## PECD capacity factors: national, all of Europe

`pecd/pecd_country_capacity_factors_simple.parquet` -- same time range as
the Germany file, with `(technology, country)` MultiIndex columns. Only
three (blended) technologies; not every country has every technology.

In [6]:
cf_eu = pd.read_parquet(paths.pecd_capacity_factors_national_europe)
cf_eu.head()

technology          wind_onshore                                           ...     solar                              
country                       AT   BA        BE        BG      CH      CY  ...        SY   TN        TR   UA   UK   XK
timestamp                                                                  ...                                        
1980-01-01 00:00:00     0.657658  0.0  0.473184  0.518195  0.5254  0.1900  ...  0.000000  0.0  0.000000  0.0  0.0  0.0
1980-01-01 01:00:00     0.673304  0.0  0.395531  0.492913  0.4753  0.1953  ...  0.000000  0.0  0.000000  0.0  0.0  0.0
1980-01-01 02:00:00     0.657121  0.0  0.376942  0.490818  0.4676  0.2141  ...  0.000000  0.0  0.000000  0.0  0.0  0.0
1980-01-01 03:00:00     0.664901  0.0  0.335175  0.840843  0.4246  0.2432  ...  0.000000  0.0  0.000000  0.0  0.0  0.0
1980-01-01 04:00:00     0.686584  0.0  0.335443  0.899662  0.4069  0.3010  ...  0.001542  0.0  0.000515  0.0  0.0  0.0

[5 rows x 109 columns]

In [7]:
timeseries_meta(cf_eu)

,value
start,1980-01-01 00:00:00
end,2025-12-31 22:00:00
rows,403247
years,46
time step (most common),0 days 01:00:00
irregular steps,0
duplicate timestamps,0
columns,109
missing values (total),0
columns with any NaN,0


In [8]:
countries_per_tech = cf_eu.columns.to_frame(index=False).groupby("technology", sort=False)["country"]
pd.DataFrame(
    {
        "n_countries": countries_per_tech.size(),
        "countries": countries_per_tech.agg(", ".join),
    }
)

,n_countries,countries
technology,,
wind_onshore,45,"AT, BA, BE, BG, CH, CY, CZ, DE, DK, DZ, EE, EG..."
wind_offshore,12,"BE, DE, DK, ES, FI, FR, IE, NL, NO, PT, SE, UK"
solar,52,"AL, AT, BA, BE, BG, CH, CY, CZ, DE, DK, DZ, EE..."


Sanity check: the Germany columns in the all-Europe file should match the
Germany-only file.

In [9]:
(cf_eu.xs("DE", axis=1, level="country")[cf_de.columns] - cf_de).abs().max()

technology
wind_onshore     0.0
wind_offshore    0.0
solar            0.0
dtype: float64

## PECD capacity factors: zonal, Germany (Track A)

Only 2015-2025, but at sub-national resolution.

### Wind onshore (PEON zones)

In [10]:
cf_on = pd.read_parquet(paths.pecd_capacity_factors_zonal_wind_onshore)
cf_on.head()

,DE01,DE02,DE03,DE04,DE05,DE06,DE07
timestamp,,,,,,,
2015-01-01 00:00:00,0.3528,0.3810,0.1916,0.2074,0.0208,0.0415,0.0501
2015-01-01 01:00:00,0.3636,0.3958,0.1991,0.2277,0.0174,0.0371,0.0487
2015-01-01 02:00:00,0.3741,0.3983,0.2001,0.2381,0.0130,0.0256,0.0349
2015-01-01 03:00:00,0.4001,0.3825,0.2144,0.2146,0.0085,0.0180,0.0240
2015-01-01 04:00:00,0.4679,0.3558,0.2264,0.1971,0.0113,0.0160,0.0189


In [11]:
timeseries_meta(cf_on)

,value
start,2015-01-01 00:00:00
end,2025-12-31 23:00:00
rows,96432
years,11
time step (most common),0 days 01:00:00
irregular steps,0
duplicate timestamps,0
columns,7
missing values (total),0
columns with any NaN,0


### Wind offshore (PEOF zones)

In [12]:
cf_off = pd.read_parquet(paths.pecd_capacity_factors_zonal_wind_offshore)
cf_off.head()

,DE011_OFF,DE012_OFF,DE013_OFF,DE014_OFF,DE015_OFF,DE02_OFF
timestamp,,,,,,
2015-01-01 00:00:00,0.812461,0.8852,NaN,NaN,NaN,0.849569
2015-01-01 01:00:00,0.826483,0.8893,NaN,NaN,NaN,0.857253
2015-01-01 02:00:00,0.836897,0.8912,NaN,NaN,NaN,0.860100
2015-01-01 03:00:00,0.860037,0.8991,NaN,NaN,NaN,0.883178
2015-01-01 04:00:00,0.866563,0.8909,NaN,NaN,NaN,0.877948


In [13]:
timeseries_meta(cf_off)

,value
start,2015-01-01 00:00:00
end,2025-12-31 23:00:00
rows,96432
years,11
time step (most common),0 days 01:00:00
irregular steps,0
duplicate timestamps,0
columns,6
missing values (total),289296
columns with any NaN,3


Missing values per offshore zone -- some zones only have data for part of
the period:

In [14]:
pd.DataFrame(
    {
        "missing_hours": cf_off.isna().sum(),
        "first_valid": cf_off.apply(pd.Series.first_valid_index),
        "last_valid": cf_off.apply(pd.Series.last_valid_index),
    }
)

,missing_hours,first_valid,last_valid
DE011_OFF,0,2015-01-01,2025-12-31 23:00:00
DE012_OFF,0,2015-01-01,2025-12-31 23:00:00
DE013_OFF,96432,NaT,NaT
DE014_OFF,96432,NaT,NaT
DE015_OFF,96432,NaT,NaT
DE02_OFF,0,2015-01-01,2025-12-31 23:00:00


### Solar (4 PV sub-technologies x NUTS2 regions)

Technology codes: 60 = industrial rooftop, 61 = residential rooftop,
62 = utility-scale fixed, 63 = utility-scale tracking. Not blended -- one
column per `(technology, region)`.

In [15]:
cf_pv = pd.read_parquet(paths.pecd_capacity_factors_zonal_solar)
cf_pv.head()

technology            60                           ...   63                         
region              DE11 DE12 DE13 DE14 DE21 DE22  ... DED2 DED4 DED5 DEE0 DEF0 DEG0
timestamp                                          ...                              
2014-12-31 23:00:00  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0  0.0  0.0
2015-01-01 00:00:00  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0  0.0  0.0
2015-01-01 01:00:00  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0  0.0  0.0
2015-01-01 02:00:00  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0  0.0  0.0
2015-01-01 03:00:00  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0  0.0  0.0

[5 rows x 152 columns]

In [16]:
timeseries_meta(cf_pv)

,value
start,2014-12-31 23:00:00
end,2025-12-31 22:00:00
rows,96432
years,12
time step (most common),0 days 01:00:00
irregular steps,0
duplicate timestamps,0
columns,152
missing values (total),0
columns with any NaN,0


In [17]:
# Mean capacity factor per technology (averaged over regions and hours)
cf_pv.mean().groupby(level="technology").mean().rename("mean_cf").to_frame()

,mean_cf
technology,
60,0.101868
61,0.112782
62,0.115447
63,0.124597


In [18]:
print("NUTS2 regions:", ", ".join(cf_pv.columns.get_level_values("region").unique()))

NUTS2 regions: DE11, DE12, DE13, DE14, DE21, DE22, DE23, DE24, DE25, DE26, DE27, DE30, DE40, DE50, DE60, DE71, DE72, DE73, DE80, DE91, DE92, DE93, DE94, DEA1, DEA2, DEA3, DEA4, DEA5, DEB1, DEB2, DEB3, DEC0, DED2, DED4, DED5, DEE0, DEF0, DEG0


## PECD region masks

Rasterized zone-membership fractions on a 0.25° grid, all of Europe.
Shown here: dimensions, number of regions, and the German zones.

In [19]:
for name, p in [("PEON (onshore)", paths.pecd_region_mask_peon), ("PEOF (offshore)", paths.pecd_region_mask_peof)]:
    with xr.open_dataset(p) as ds:
        regions = ds["region"].values.astype(str)
        de = [r for r in regions if r.startswith("DE")]
        print(f"--- {name}: {p.name}")
        print(f"dims: {dict(ds.sizes)}")
        print(f"lat: {float(ds.latitude.min())} .. {float(ds.latitude.max())}, "
              f"lon: {float(ds.longitude.min())} .. {float(ds.longitude.max())}")
        print(f"regions: {len(regions)} total, {len(de)} German: {', '.join(de)}")
        print(f"attrs: {ds.attrs}\n")

--- PEON (onshore): peon_region_mask.nc
dims: {'region': 153, 'latitude': 229, 'longitude': 305}
lat: 18.0 .. 75.0, lon: -31.0 .. 45.0
regions: 153 total, 7 German: DE01, DE02, DE03, DE04, DE05, DE06, DE07
attrs: {'title': 'Pan european on-shore region mask', 'institution': 'Copernicus Climate Change Service, Sectoral Information System for the Energy Sector', 'source': 'shape file: DTU-PECD40-Polygons_VF20230127 by ENTSO-E (https://transparency.entsoe.eu/)'}

--- PEOF (offshore): peof_region_mask.nc
dims: {'region': 124, 'latitude': 229, 'longitude': 305}
lat: 18.0 .. 75.0, lon: -31.0 .. 45.0
regions: 124 total, 6 German: DE011_OFF, DE012_OFF, DE02_OFF, DE015_OFF, DE014_OFF, DE013_OFF
attrs: {'title': 'Pan european off-shore region mask', 'institution': 'Copernicus Climate Change Service, Sectoral Information System for the Energy Sector', 'source': 'shape file: PECD42_VF20241002'}



In [20]:
with xr.open_dataset(paths.pecd_region_mask_peon) as ds:
    de_regions = [r for r in ds["region"].values.astype(str) if r.startswith("DE")]
    mask_de = ds["mask"].sel(region=de_regions).to_dataframe().query("mask > 0")
mask_de.head()

mask
region latitude longitude          
DE01   55.0     8.25       0.124260
                8.50       0.174556
                8.75       0.090237
                9.00       0.068047
                9.50       0.001479

## MaStR unit-level records (Track A)

Loaded column-wise; only the columns needed for the metadata are read
for the large solar table.

In [21]:
def mastr_meta(path):
    cols = ["commissioning_date", "final_shutdown_date", "net_capacity_kw", "latitude"]
    df = pd.read_parquet(path, columns=cols)
    real_dates = df["commissioning_date"][df["commissioning_date"] > "1900-01-01"]
    return pd.Series(
        {
            "units": len(df),
            "commissioning_date min (excl. 1900-01-01)": real_dates.min(),
            "commissioning_date max": df["commissioning_date"].max(),
            "commissioning_date == 1900-01-01": int((df["commissioning_date"] == "1900-01-01").sum()),
            "commissioning_date missing": int(df["commissioning_date"].isna().sum()),
            "units with final_shutdown_date": int(df["final_shutdown_date"].notna().sum()),
            "units with coordinates": int(df["latitude"].notna().sum()),
            "total net capacity (GW, incl. shut down)": round(df["net_capacity_kw"].sum() / 1e6, 1),
        },
        name="value",
    ).to_frame()

### Solar units

In [22]:
head(paths.mastr_solar_units)

,unit_id,energy_source,state,district,municipality_key,postal_code,...,unit_system_status,feed_in_type,location_id,usage_sector,installation_type,technology
0,SEE984033548619,Solare Strahlungsenergie,Nordrhein-Westfalen,Münster,05515000,48147,...,Aktiviert,Volleinspeisung,SEL948991715391,Haushalt,Gebäudesolaranlage,solar
1,SEE901901460125,Solare Strahlungsenergie,Baden-Württemberg,Ostalbkreis,08136065,73529,...,Aktiviert,Teileinspeisung (einschließlich Eigenverbrauch),SEL982068309366,Haushalt,Gebäudesolaranlage,solar
2,SEE983679054270,Solare Strahlungsenergie,Brandenburg,Havelland,12063208,14641,...,Aktiviert,Teileinspeisung (einschließlich Eigenverbrauch),SEL906699064968,Haushalt,Gebäudesolaranlage,solar
3,SEE978732598938,Solare Strahlungsenergie,Bayern,Regensburg,09375180,93080,...,Aktiviert,Teileinspeisung (einschließlich Eigenverbrauch),SEL996128012264,Haushalt,Gebäudesolaranlage,solar
4,SEE970592691989,Solare Strahlungsenergie,Saarland,Saarlouis,10044115,66740,...,Aktiviert,Teileinspeisung (einschließlich Eigenverbrauch),SEL975715515692,Haushalt,Gebäudesolaranlage,solar


In [23]:
mastr_meta(paths.mastr_solar_units)

,value
units,6304840
commissioning_date min (excl. 1900-01-01),1900-02-24 00:00:00
commissioning_date max,2026-07-16 00:00:00
commissioning_date == 1900-01-01,23
commissioning_date missing,67939
units with final_shutdown_date,76116
units with coordinates,261409
"total net capacity (GW, incl. shut down)",120.599998


In [24]:
pd.read_parquet(paths.mastr_solar_units, columns=["installation_type"])["installation_type"].value_counts(dropna=False).to_frame()

,count
installation_type,
Gebäudesolaranlage,4780090
Steckerfertige Solaranlage (sog. Balkonkraftwerk),1503907
Freiflächensolaranlage,20711
Sonstige Solaranlage,132


### Solar technical detail (orientation / tilt)

In [25]:
head(paths.mastr_solar_technical_detail)

,unit_id,main_orientation,main_orientation_tilt_bucket
0,SEE984033548619,Süd,21 - 40 Grad
1,SEE901901460125,Süd,21 - 40 Grad
2,SEE983679054270,Süd,21 - 40 Grad
3,SEE978732598938,Süd-West,21 - 40 Grad
4,SEE970592691989,West,21 - 40 Grad


In [26]:
detail = pd.read_parquet(paths.mastr_solar_technical_detail, columns=["main_orientation", "main_orientation_tilt_bucket"])
pd.concat(
    {c: detail[c].value_counts(dropna=False) for c in detail.columns}, axis=1
)

,main_orientation,main_orientation_tilt_bucket
Süd,2350251.0,NaN
NaN,1255152.0,1289353.0
Süd-West,933266.0,NaN
Süd-Ost,638218.0,NaN
Ost-West,401462.0,NaN
West,337721.0,NaN
Ost,263685.0,NaN
Nord-Ost,44939.0,NaN
Nord-West,38630.0,NaN
Nord,35659.0,NaN


In [27]:
del detail

### Wind units

In [28]:
wind = pd.read_parquet(paths.mastr_wind_units)
wind.head()

,unit_id,energy_source,state,district,municipality_key,postal_code,...,resumption_of_operation_date,unit_operational_status,unit_system_status,wind_onshore_or_offshore,sea_location,technology
0,SEE940146675093,Wind,Hessen,Werra-Meißner-Kreis,06636200,34298,...,NaT,In Betrieb,Aktiviert,Windkraft an Land,NaN,wind
1,SEE973767078653,Wind,Schleswig-Holstein,Segeberg,01060017,23824,...,NaT,In Betrieb,Aktiviert,Windkraft an Land,NaN,wind
2,SEE914108319653,Wind,Hessen,Werra-Meißner-Kreis,06636200,34298,...,NaT,In Betrieb,Aktiviert,Windkraft an Land,NaN,wind
3,SEE982417853618,Wind,Hessen,Werra-Meißner-Kreis,06636200,34298,...,NaT,In Betrieb,Aktiviert,Windkraft an Land,NaN,wind
4,SEE913741454097,Wind,Nordrhein-Westfalen,Heinsberg,05370016,52525,...,NaT,In Betrieb,Aktiviert,Windkraft an Land,NaN,wind


In [29]:
mastr_meta(paths.mastr_wind_units)

,value
units,43262
commissioning_date min (excl. 1900-01-01),1949-07-31 00:00:00
commissioning_date max,2026-07-14 00:00:00
commissioning_date == 1900-01-01,0
commissioning_date missing,8146
units with final_shutdown_date,2926
units with coordinates,41987
"total net capacity (GW, incl. shut down)",135.399994


In [30]:
wind.groupby("wind_onshore_or_offshore", observed=True).agg(
    units=("unit_id", "size"),
    net_capacity_GW=("net_capacity_kw", lambda s: round(s.sum() / 1e6, 1)),
)

,units,net_capacity_GW
wind_onshore_or_offshore,,
Windkraft an Land,41353,122.400002
Windkraft auf See,1909,13.000000


## Region crosswalk: LAU → NUTS3

In [31]:
lau = pd.read_parquet(paths.lau_nuts_correspondence)
lau.head()

,municipality_key,nuts3_code
0,08111000,DE111
1,08115001,DE112
2,08115002,DE112
3,08115003,DE112
4,08115004,DE112


In [32]:
pd.Series(
    {
        "rows": len(lau),
        "unique municipality_key": lau["municipality_key"].nunique(),
        "unique NUTS3": lau["nuts3_code"].nunique(),
        "unique NUTS2 (first 4 chars)": lau["nuts3_code"].str[:4].nunique(),
    },
    name="value",
).to_frame()

,value
rows,10980
unique municipality_key,10980
unique NUTS3,401
unique NUTS2 (first 4 chars),39
